# 4D SfM — full batch (all dates)

The production run: processes **every dated image set** found in the time-lapse
directory, across the entire record (multi-year capable — not limited to one
year). Same per-date workflow as `4d_sfm_dem_monthly.ipynb`, which instead
processes one hand-picked date per month.

For each date it produces DEM + orthoimage + DoD + stable-terrain DoD + M3C2
raster (with histograms). All logic lives in
`tlapse4d.pipeline_4dsfm.run_4dsfm_day_with_rasters`; this notebook is just
configuration + date discovery + one loop.

> **Resume:** the per-date pipeline caches every step (`overwrite=False`), so
> re-running this notebook skips already-finished dates cheaply — safe to stop
> and restart on a long run. A date that fails (e.g. cloud-cover gate) is
> recorded as an error row and the loop continues.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
os.environ["AGISOFT_LICENSE_PATH"] = "/home/asus/.config/Agisoft/license.lic"

from pathlib import Path

import pandas as pd
import Metashape  # noqa: F401  — must import after AGISOFT_LICENSE_PATH is set
from tlapse4d.pipeline_4dsfm import run_4dsfm_day_with_rasters
from tlapse4d.metashape import discover_images, _normalize_date

## Configuration

Edit only this section. Paths come from `site_config.py`; per-stage knobs live
in `params` (identical to the monthly notebook).

In [ ]:
# ── Site — edit the 3 paths in site_config.py to choose / switch glacier ─
import site_config_north as site

# ── Per-date knobs (SfM + raster combined) ───────────────────────────
params = dict(
    # SfM pipeline knobs (forwarded to run_4dsfm_day)
    match_downscale       = 1,
    depth_downscale       = 2,
    filter_mode           = "Mild",        # "Mild" or "Aggressive"
    loc_acc_new           = (0.5, 0.5, 0.5),
    rot_acc_new           = (5.0, 5.0, 5.0),
    ref_downsample        = 0.4,   # per-glacier coreg knob (0.05 North dense / 0.40 West)
    tba_downsample        = 1.0,
    p2p_max_disp          = 10,
    sp2p_max_disp         = 5,
    m_sp2p_max_disp       = 2,
    p2p_outlier_ratio     = 0.75,
    sp2p_outlier_ratio    = 0.75,
    m_sp2p_outlier_ratio  = 0.75,  # lower to 0.5-0.6 to tighten Stage 3 vs glacier false matches
    use_ecef              = True,
    overwrite             = False,
    verbose               = False,
    # Registry frozen at the 2023-11-27 baseline (no feedback into BA).
    add_to_registry       = False,
    # Skip Step 6 rebuild + Step 6b validation (coreg M3C2 plot still runs).
    run_validation        = False,
    # Cloud-cover gate (now in Step 1 / 4D SfM): skip the date if >= this many
    # NEW-DAY cameras fail to align in the multi-temporal bundle adjustment.
    max_unaligned         = 6,
    # Keep only daytime frames; drop off-schedule night / motion captures BEFORE
    # the alignment gate counts them. None = keep every frame.
    time_window           = (9, 17),
    exclude_cameras       = [{"camera": "C8", "from": "2024-07-15"}, {"camera": "C9", "from": "2024-07-15"}],  # drop boulder-degraded C8+C9 from 07-15

    # Raster knobs
    dem_method            = "point2dem",   # HSfM ASP point2dem (IDW); "cubic" = legacy
    res                   = 1.0,
    max_gap_pixels        = 1,
    ref_cloud_downsample  = 0.25,
    m3c2_ref_downsample   = 0.25,
    slope_threshold       = 60.0,
    overwrite_ref_dem     = False,
    overwrite_day_dem     = False,
    overwrite_dod         = False,
    overwrite_stable      = False,
    overwrite_stable_dod  = False,
    overwrite_m3c2        = False,
)

# Optional inclusive date bounds ("YYYY-MM-DD"); set either to None for no bound.
# Leave both None to process every discovered date (the full record).
date_from = "2024-07-15"
date_to   = None

## Discover dates

Scans the time-lapse directory for every standardised image
(`<cam>_<YYYY-MM-DD>_<HHMMSS>.<ext>`), drops the reference/baseline day(s)
(read from the registry — those are never reprocessed as TBA days), and applies
the optional `date_from` / `date_to` bounds. Edit `dates` directly below for a
custom subset.

In [ ]:
# Every date present in the time-lapse imagery (sorted ascending).
all_dates = sorted(discover_images(site.tlcam_dir).keys())

# Reference/baseline days live in the registry — exclude them as TBA days.
ref_dates = set(pd.read_csv(site.registry_csv)["date"].map(_normalize_date))

# ISO YYYY-MM-DD strings sort/compare chronologically, so >=/<= bound directly.
dates = [
    d for d in all_dates
    if d not in ref_dates
    and (date_from is None or d >= date_from)
    and (date_to   is None or d <= date_to)
]

print(f"Discovered {len(all_dates)} dated image set(s); "
      f"{len(ref_dates)} reference day(s) excluded.")
print(f"{len(dates)} date(s) to process"
      + (f": {dates[0]} … {dates[-1]}" if dates else "."))
dates

## Run

Loops over the discovered `dates` and calls `run_4dsfm_day_with_rasters`
for each, wrapped in `try / except` so one bad date doesn't stop the batch.
Per-date stats are collected, printed, and written to a CSV at the end.

In [ ]:
import traceback

summary = []
for d in dates:
    print(f"\n{'#'*70}\n#  {d}\n{'#'*70}")
    try:
        summary.append(run_4dsfm_day_with_rasters(
            new_date     = d,
            tlcam_dir    = site.tlcam_dir,
            ref_cloud    = site.ref_cloud,
            glacier_mask = site.glacier_mask,
            registry_csv = site.registry_csv,
            output_dir   = site.output_dir,
            **params,
        ))
    except Exception as e:
        print(f"\n  {d} failed: {type(e).__name__}: {e}")
        traceback.print_exc()
        summary.append({"date": d, "error": repr(e)})

print(f"\n{'='*70}\n  Batch summary ({len(summary)} dates)\n{'='*70}")
for s in summary:
    if "error" in s:
        print(f"  {s['date']} : ERROR — {s['error']}")
    else:
        print(
            f"  {s['date']} : "
            f"DoD med={s['dod_stats']['median']:+.2f} m  std={s['dod_stats']['std']:.2f}  |  "
            f"stable med={s['stable_stats']['median']:+.2f} m  std={s['stable_stats']['std']:.2f}  |  "
            f"M3C2 med={s['m3c2_stats']['median']:+.2f} m  std={s['m3c2_stats']['std']:.2f}"
        )

## Save summary

Flattens the per-date stats into a CSV under `output/` for the whole batch.

In [ ]:
rows = []
for s in summary:
    if "error" in s:
        rows.append({"date": s["date"], "error": s["error"]})
    else:
        rows.append({
            "date":          s["date"],
            "dod_median":    s["dod_stats"]["median"],    "dod_std":    s["dod_stats"]["std"],
            "stable_median": s["stable_stats"]["median"], "stable_std": s["stable_stats"]["std"],
            "m3c2_median":   s["m3c2_stats"]["median"],   "m3c2_std":   s["m3c2_stats"]["std"],
        })

df = pd.DataFrame(rows)
out_csv = site.output_dir / "output" / "batch_summary.csv"
df.to_csv(out_csv, index=False)
print(f"Saved {out_csv}")
df